# BBC News scraper: headlines and descriptions

This notebook fetches the current BBC News homepage, extracts article headlines and descriptions, removes duplicates, previews the data, and saves it as a CSV file.

> Please respect BBC's terms of use, robots.txt, rate limits, and copyright. This example is for educational, low-volume scraping only.

In [3]:
from datetime import datetime, timezone
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BBC_URL = "https://www.bbc.com/news"
OUTPUT_FILE = "bbc_news_today.csv"

response = requests.get(
    BBC_URL,
    headers={"User-Agent": "Mozilla/5.0 (compatible; educational-news-scraper/1.0)"},
    timeout=20,
)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
retrieved_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
print(f"Fetched {BBC_URL} at {retrieved_at}")
print(f"HTTP status: {response.status_code}")

Fetched https://www.bbc.com/news at 2026-09-28T17:14:47+00:00
HTTP status: 200


In [4]:
def clean_text(value):
    return " ".join(value.split()) if value else ""


def find_description(heading):
    heading_text = clean_text(heading.get_text(" ", strip=True))

    # Search only until the next headline so one story cannot borrow another story's text.
    for element in heading.next_elements:
        if getattr(element, "name", None) in {"h2", "h3"}:
            break
        if getattr(element, "name", None) != "p":
            continue

        description = clean_text(element.get_text(" ", strip=True))
        if description and description != heading_text:
            return description

    return ""


records = []
seen_urls = set()

for heading in soup.select("h2, h3"):
    headline = clean_text(heading.get_text(" ", strip=True))
    link = heading.find_parent("a", href=True)

    if not headline or link is None:
        continue

    article_url = urljoin(BBC_URL, link["href"])
    if "/news/" not in article_url or article_url in seen_urls:
        continue

    seen_urls.add(article_url)
    records.append(
        {
            "headline": headline,
            "description": find_description(heading),
            "url": article_url,
            "retrieved_at_utc": retrieved_at,
        }
    )

news_df = pd.DataFrame(records).drop_duplicates(subset="url").reset_index(drop=True)

if news_df.empty:
    raise RuntimeError(
        "No BBC articles were found. The page markup may have changed, or access may be blocked."
    )

print(f"Extracted {len(news_df)} unique BBC articles")
print(f"Unique descriptions: {news_df['description'].nunique()} of {len(news_df)} rows")
news_df.head(10)

Extracted 35 unique BBC articles
Unique descriptions: 22 of 35 rows


,headline,description,url,retrieved_at_utc
0,UK military base suspects to be released on ba...,Laurence Taylor says the five men arrested on ...,https://www.bbc.com/news/live/c65y7nl29k0et,2026-09-28T17:14:47+00:00
1,What we know about military base counter-terro...,Five men have been arrested as counter-terror ...,https://www.bbc.com/news/articles/c85ydnwqpzyzo,2026-09-28T17:14:47+00:00
2,Inside Yemen's front-line city as Houthis batt...,In rare access to Yemen's conflict zone the BB...,https://www.bbc.com/news/articles/cw98005ndz7no,2026-09-28T17:14:47+00:00
3,Protestant Orange Order parade dispute continu...,A controversial march was due to take place on...,https://www.bbc.com/news/articles/cmqlwn60yyq9o,2026-09-28T17:14:47+00:00
4,Trump-Xi summit: What wasn't said might matter...,The most revealing thing about the summit in W...,https://www.bbc.com/news/articles/cxp84g2ly1mjo,2026-09-28T17:14:47+00:00
5,SpaceX's Starship splashes down in Pacific Oce...,The rocket - which had been due to orbit the p...,https://www.bbc.com/news/live/c65ymmpzev1mt,2026-09-28T17:14:47+00:00
6,Seoul summons Ukraine envoy over North Korean ...,Last week Zelensky revealed Ukraine had sent t...,https://www.bbc.com/news/articles/c8ly40xx0dr0o,2026-09-28T17:14:47+00:00
7,Plan for controversial Sydney data centre scra...,"Goodman Group, the property giant behind the P...",https://www.bbc.com/news/articles/c6n9wrqpj4yqo,2026-09-28T17:14:47+00:00
8,"Three dead, including child, in English Channe...",,https://www.bbc.com/news/articles/c6vgy71rle35o,2026-09-28T17:14:47+00:00
9,Four bodies found after avalanche hits Himalay...,,https://www.bbc.com/news/articles/cwjdvml9e897o,2026-09-28T17:14:47+00:00


In [ ]:
# Keep headline and description columns first for easier analysis.
news_df = news_df[["headline", "description", "url", "retrieved_at_utc"]]


try:
    news_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
except PermissionError:
    fallback_file = f"bbc_news_today_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    news_df.to_csv(fallback_file, index=False, encoding="utf-8-sig")
    OUTPUT_FILE = fallback_file
    print(f"The original CSV is locked, so the data was saved to {OUTPUT_FILE}")
else:
    print(f"Saved {len(news_df)} rows to {OUTPUT_FILE}")

news_df.loc[:, ["headline", "description"]].head(10)

Saved 35 rows to bbc_news_today.csv


,headline,description
0,UK military base suspects to be released on ba...,Laurence Taylor says the five men arrested on ...
1,What we know about military base counter-terro...,Five men have been arrested as counter-terror ...
2,Inside Yemen's front-line city as Houthis batt...,In rare access to Yemen's conflict zone the BB...
3,Protestant Orange Order parade dispute continu...,A controversial march was due to take place on...
4,Trump-Xi summit: What wasn't said might matter...,The most revealing thing about the summit in W...
5,SpaceX's Starship splashes down in Pacific Oce...,The rocket - which had been due to orbit the p...
6,Seoul summons Ukraine envoy over North Korean ...,Last week Zelensky revealed Ukraine had sent t...
7,Plan for controversial Sydney data centre scra...,"Goodman Group, the property giant behind the P..."
8,"Three dead, including child, in English Channe...",
9,Four bodies found after avalanche hits Himalay...,


## How this works

1. `requests.get()` downloads the page HTML.
2. `BeautifulSoup` parses the HTML document.
3. `h2` and `h3` elements provide likely article headlines.
4. The nearest article/card is searched for a paragraph description.
5. Article URLs are normalized, deduplicated, and exported to `bbc_news_today.csv`.

Run the cells from top to bottom each time you want a fresh snapshot. The output represents the page at retrieval time, not a guaranteed archive of every story published today. If BBC changes its HTML, inspect `response.text` and update the selectors.